EDA

In [ ]:
import cv2
import os
import numpy as np
import mediapipe as mp
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Inisialisasi MediaPipe
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(static_image_mode=True, refine_landmarks=True)

# Path ke dataset
root_folder = 'D:\Kuliah\Skripsi\Data'

# Warna tetap per kategori
colors = {
    "frontal_terang": "#66c2a5",
    "frontal_gelap": "#fc8d62",
    "nonfrontal_terang": "#8da0cb",
    "nonfrontal_gelap": "#e78ac3",
    "tidak_terdeteksi": "#a6d854"
}

# List semua kategori
all_categories = list(colors.keys())

# Data akhir
all_class_stats = []

# Fungsi frontal
def is_frontal(landmarks):
    left_eye = landmarks[33]
    right_eye = landmarks[263]
    nose_tip = landmarks[1]
    mid_x = (left_eye.x + right_eye.x) / 2
    offset = abs(nose_tip.x - mid_x)
    return offset < 0.03

# Fungsi terang/gelap
def is_bright(image_gray):
    return np.mean(image_gray) > 100

# Telusuri setiap kelas
for class_name in os.listdir(root_folder):
    class_path = os.path.join(root_folder, class_name)
    if not os.path.isdir(class_path):
        continue

    count = {
        "class": class_name,
        "frontal_terang": 0,
        "frontal_gelap": 0,
        "nonfrontal_terang": 0,
        "nonfrontal_gelap": 0,
        "tidak_terdeteksi": 0
    }

    for filename in os.listdir(class_path):
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_path = os.path.join(class_path, filename)
            image = cv2.imread(img_path)
            if image is None:
                continue

            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image_gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            results = face_mesh.process(image_rgb)
            bright = is_bright(image_gray)

            if results.multi_face_landmarks:
                landmarks = results.multi_face_landmarks[0].landmark
                frontal = is_frontal(landmarks)

                if frontal and bright:
                    count['frontal_terang'] += 1
                elif frontal and not bright:
                    count['frontal_gelap'] += 1
                elif not frontal and bright:
                    count['nonfrontal_terang'] += 1
                else:
                    count['nonfrontal_gelap'] += 1
            else:
                count['tidak_terdeteksi'] += 1

    # Hitung total dan rasio
    total = sum(count[k] for k in all_categories)
    for k in all_categories:
        count[f'rasio_{k}'] = (count[k] / total * 100) if total > 0 else 0

    all_class_stats.append(count)

    # Pie chart per kelas
    pie_sizes = [count[k] for k in all_categories]
    if sum(pie_sizes) > 0:
        pie_labels = [
            'Frontal Terang', 'Frontal Gelap',
            'Nonfrontal Terang', 'Nonfrontal Gelap',
            'Tidak Terdeteksi'
        ]
        plt.figure(figsize=(5, 5))
        plt.pie(
            pie_sizes,
            labels=pie_labels,
            autopct='%1.1f%%',
            startangle=140,
            colors=[colors[k] for k in all_categories]
        )
        plt.title(f'Pie Chart - {class_name}')
        plt.tight_layout()
        plt.savefig(f"piechart_{class_name}.png")
        plt.close()

# Simpan ke Excel
df_result = pd.DataFrame(all_class_stats)
excel_path = "hasil_klasifikasi_lengkap.xlsx"
df_result.to_excel(excel_path, index=False)
print(f"\n✅ Data disimpan ke: {excel_path}")

# Bar chart semua kategori termasuk tidak terdeteksi
df_plot = df_result.set_index('class')[all_categories]
df_plot.plot(
    kind='bar',
    stacked=True,
    figsize=(12, 6),
    color=[colors[k] for k in all_categories]
)
plt.title("Distribusi Pose, Cahaya, dan Deteksi Wajah per Kelas")
plt.xlabel("Kelas (Subfolder)")
plt.ylabel("Jumlah Gambar")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("bar_chart_dengan_tidak_terdeteksi.png")
plt.show()

# === Pie Chart Global (semua kelas digabung) ===
total_global = {k: 0 for k in all_categories}
for stat in all_class_stats:
    for k in all_categories:
        total_global[k] += stat[k]

# Buat pie chart global
global_sizes = [total_global[k] for k in all_categories]
global_labels = [
    'Frontal Terang', 'Frontal Gelap',
    'Nonfrontal Terang', 'Nonfrontal Gelap',
    'Tidak Terdeteksi'
]

if sum(global_sizes) > 0:
    plt.figure(figsize=(6, 6))
    plt.pie(
        global_sizes,
        labels=global_labels,
        autopct='%1.1f%%',
        startangle=140,
        colors=[colors[k] for k in all_categories]
    )
    plt.title("Pie Chart Global - Distribusi Semua Kelas")
    plt.tight_layout()
    plt.savefig("piechart_global.png")
    plt.show()

# === Load Data dari Excel ===
file_path = "hasil_klasifikasi_lengkap.xlsx"
df = pd.read_excel(file_path)

# === Warna Kategori ===
colors = {
    'Frontal Terang': '#4CAF50',      # Hijau
    'Frontal Gelap': '#FFEB3B',       # Kuning
    'Nonfrontal Terang': '#2196F3',   # Biru
    'Nonfrontal Gelap': '#F44336',    # Merah
    'Tidak Terdeteksi': '#9E9E9E'     # Abu
}

sns.set(style="whitegrid")

# === Bar Plot Global (Keseluruhan) ===
total_counts = {
    'Frontal Terang': df['frontal_terang'].sum(),
    'Frontal Gelap': df['frontal_gelap'].sum(),
    'Nonfrontal Terang': df['nonfrontal_terang'].sum(),
    'Nonfrontal Gelap': df['nonfrontal_gelap'].sum(),
    'Tidak Terdeteksi': df['tidak_terdeteksi'].sum()
}

plt.figure(figsize=(10, 7))
bars = plt.bar(
    list(total_counts.keys()),
    list(total_counts.values()),
    color=[colors[k] for k in total_counts.keys()],
    width=0.6
)
plt.title('Distribusi Kategori Keseluruhan', fontsize=16)
plt.ylabel('Jumlah Gambar', fontsize=13)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig("barplot_global.png")
plt.show()

PROSES SPLIT DATASET

In [ ]:
import os
import matplotlib.pyplot as plt
import shutil
import random
from pathlib import Path

# Path folder original dengan 40 subfolder (kelas)
dataset_path = Path('D:/Kuliah/Skripsi/Data')  

# Folder baru untuk train/val split
output_dir = Path('D:/Kuliah/Skripsi/Data_StratifiedSplit')
train_dir = output_dir / 'train'
val_dir = output_dir / 'val'

# Proporsi split
val_ratio = 0.25

if train_dir.exists():
    shutil.rmtree(train_dir)
if val_dir.exists():
    shutil.rmtree(val_dir)

train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)

# Ambil nama kelas (subfolder)
class_names = sorted([d.name for d in dataset_path.iterdir() if d.is_dir()])

# Proses stratified split
for class_name in class_names:
    class_folder = dataset_path / class_name
    images = list(class_folder.glob('*'))
    random.shuffle(images)
    
    n_val = int(len(images) * val_ratio)
    
    # Buat folder kelas di train dan val
    (train_dir / class_name).mkdir(parents=True, exist_ok=True)
    (val_dir / class_name).mkdir(parents=True, exist_ok=True)
    
    # Pindah/Copy file ke train dan val folder
    for i, img_path in enumerate(images):
        if i < n_val:
            dest = val_dir / class_name / img_path.name
        else:
            dest = train_dir / class_name / img_path.name
        shutil.copy(img_path, dest)

print("✅ Stratified split selesai dibuat di:", output_dir)

#PLOT
# Path dataset
original_dir = 'D:/Kuliah/Skripsi/Data'
preprocessed_dir = 'D:/Kuliah/Skripsi/Data_StratifiedSplit'
train_dir = os.path.join(preprocessed_dir, 'train')
val_dir = os.path.join(preprocessed_dir, 'val')

def count_images_per_class(directory):
    counts = {}
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            num_files = len([
                f for f in os.listdir(class_path)
                if os.path.isfile(os.path.join(class_path, f))
            ])
            counts[class_name] = num_files
    return counts

# --------- 1. Hitung data SEBELUM split ---------
original_counts = {}
original_dir_path = Path(original_dir)
class_names = sorted([d.name for d in original_dir_path.iterdir() if d.is_dir()])
for class_name in class_names:
    class_path = original_dir_path / class_name
    original_counts[class_name] = len(list(class_path.glob('*')))

# --------- 2. Hitung data SETELAH split ---------
train_counts = count_images_per_class(train_dir)
val_counts = count_images_per_class(val_dir)

# --------- 3. 40 warna unik ---------
colors_tab20 = list(plt.get_cmap('tab20').colors)
colors_tab20b = list(plt.get_cmap('tab20b').colors)
colors_tab20c = list(plt.get_cmap('tab20c').colors)
unique_colors = (colors_tab20 + colors_tab20b + colors_tab20c)[:40]

# --------- 4. Visualisasi ---------
# Plot: Sebelum split
plt.figure(figsize=(12, 6))
plt.bar(list(original_counts.keys()), list(original_counts.values()), color=unique_colors)
plt.xticks(rotation=45, ha='right')
plt.xlabel("Kelas")
plt.ylabel("Jumlah Citra")
plt.title("Distribusi Data Citra Per Kelas")
plt.tight_layout()
plt.show()

# Plot: Train
plt.figure(figsize=(12, 6))
plt.bar(list(train_counts.keys()), list(train_counts.values()), color='skyblue')
plt.xticks(rotation=45, ha='right')
plt.xlabel("Kelas")
plt.ylabel("Jumlah Citra")
plt.title("Distribusi Citra Data Training")
plt.tight_layout()
plt.show()

# Plot: Validation
plt.figure(figsize=(12, 6))
plt.bar(list(val_counts.keys()), list(val_counts.values()), color='salmon')
plt.xticks(rotation=45, ha='right')
plt.xlabel("Kelas")
plt.ylabel("Jumlah Citra")
plt.title("Distribusi Citra Data Validation")
plt.tight_layout()
plt.show()

PROSES FACE ALIGNMENT & AUGMENTASI

In [ ]:
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import numpy as np
from mtcnn import MTCNN
import albumentations as A
from PIL import Image
from tqdm import tqdm
import shutil

# Path dataset hasil stratified split
input_root = Path("D:/Kuliah/Skripsi/Data_StratifiedSplit")
output_root = Path("D:/Kuliah/Skripsi/Data_Preprocessed")

# Augmentasi hanya: horizontal flip dan rotasi ±15°
augmentations = A.Compose([
    A.HorizontalFlip(p=1.0),
    A.Rotate(limit=15, p=1.0)
])

# Setup MTCNN
detector = MTCNN()

def align_and_crop_face(img, detector, output_size=(224, 224)):
    results = detector.detect_faces(img)
    if results:
        keypoints = results[0]['keypoints']
        left_eye, right_eye = keypoints['left_eye'], keypoints['right_eye']

        dy = right_eye[1] - left_eye[1]
        dx = right_eye[0] - left_eye[0]
        angle = np.degrees(np.arctan2(dy, dx))

        eyes_center = (int((left_eye[0] + right_eye[0]) / 2),
                       int((left_eye[1] + right_eye[1]) / 2))

        M = cv2.getRotationMatrix2D(eyes_center, angle, 1)
        aligned = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), flags=cv2.INTER_CUBIC)

        x, y, w, h = results[0]['box']
        x, y, w, h = int(x), int(y), int(w), int(h)

        # Validasi batas cropping
        x = max(0, x)
        y = max(0, y)
        x_end = min(x + w, aligned.shape[1])
        y_end = min(y + h, aligned.shape[0])
        face = aligned[y:y_end, x:x_end]

        if face.size == 0 or face.shape[0] < 20 or face.shape[1] < 20:
            return None

        face = cv2.resize(face, output_size)
        return face
    return None

def try_detect_with_rotation(img, detector, angles=[0, -15, 15, -30, 30, -90, 90, 180]):
    for angle in angles:
        if angle != 0:
            M = cv2.getRotationMatrix2D((img.shape[1] // 2, img.shape[0] // 2), angle, 1)
            rotated_img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))
        else:
            rotated_img = img.copy()

        result = align_and_crop_face(rotated_img, detector)
        if result is not None:
            return result
    return None

if output_root.exists():
    shutil.rmtree(output_root)
output_root.mkdir(parents=True)

# Proses semua gambar dari train dan val set
for subset in ['train', 'val']:
    subset_path = input_root / subset
    output_subset_path = output_root / subset

    for class_folder in tqdm(list(subset_path.glob("*")), desc=f"Memproses {subset}"):
        class_name = class_folder.name
        output_class_path = output_subset_path / class_name
        output_class_path.mkdir(parents=True, exist_ok=True)

        for img_path in class_folder.glob("*"):
            img = cv2.imdecode(np.fromfile(str(img_path), dtype=np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            aligned_face = try_detect_with_rotation(img_rgb, detector)
            if aligned_face is None:
                continue

            # Simpan hasil aligned asli
            base_name = img_path.stem
            save_path = output_class_path / f"{base_name}_orig.jpg"
            Image.fromarray(aligned_face).save(save_path)

            for i in range(2):  # Dua augmentasi (flip + rotate random)
                augmented = augmentations(image=aligned_face)
                aug_img = augmented['image']
                aug_path = output_class_path / f"{base_name}_aug{i+1}.jpg"
                Image.fromarray(aug_img).save(aug_path)

print("✅ Proses face alignment dan augmentasi selesai.")

# Path sebelum dan sesudah preprocessing
base_before = 'D:/Kuliah/Skripsi/Data_StratifiedSplit'
before_train_dir = os.path.join(base_before, 'train')
before_val_dir = os.path.join(base_before, 'val')

base_after = 'D:/Kuliah/Skripsi/Data_Preprocessed'
after_train_dir = os.path.join(base_after, 'train')
after_val_dir = os.path.join(base_after, 'val')

# Fungsi untuk menghitung jumlah gambar per kelas
def count_images_per_class(directory):
    counts = {}
    for class_name in sorted(os.listdir(directory)):
        class_path = os.path.join(directory, class_name)
        if os.path.isdir(class_path):
            num_files = len([
                f for f in os.listdir(class_path)
                if os.path.isfile(os.path.join(class_path, f))
            ])
            counts[class_name] = num_files
    return counts

# Hitung jumlah gambar per kelas (sebelum dan sesudah)
train_counts_before = count_images_per_class(before_train_dir)
val_counts_before = count_images_per_class(before_val_dir)
train_counts_after = count_images_per_class(after_train_dir)
val_counts_after = count_images_per_class(after_val_dir)

# ================================
# Plot Perbandingan per Kelas: TRAIN
# ================================
train_labels = sorted(set(train_counts_before.keys()).union(train_counts_after.keys()))
train_before_values = [train_counts_before.get(cls, 0) for cls in train_labels]
train_after_values = [train_counts_after.get(cls, 0) for cls in train_labels]

x = range(len(train_labels))
bar_width = 0.4

plt.figure(figsize=(14, 6))
plt.bar([i - bar_width/2 for i in x], train_before_values, width=bar_width, label='Sebelum', color='gray')
plt.bar([i + bar_width/2 for i in x], train_after_values, width=bar_width, label='Sesudah', color='skyblue')
plt.xticks(x, train_labels, rotation=45, ha='right')
plt.ylabel("Jumlah Citra")
plt.title("Perbandingan Jumlah Citra Training per Kelas (Sebelum vs Sesudah)")
plt.legend()
plt.grid(axis='y', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

# ================================
# Plot Perbandingan per Kelas: VALIDATION
# ================================
val_labels = sorted(set(val_counts_before.keys()).union(val_counts_after.keys()))
val_before_values = [val_counts_before.get(cls, 0) for cls in val_labels]
val_after_values = [val_counts_after.get(cls, 0) for cls in val_labels]

x = range(len(val_labels))

plt.figure(figsize=(14, 6))
plt.bar([i - bar_width/2 for i in x], val_before_values, width=bar_width, label='Sebelum', color='gray')
plt.bar([i + bar_width/2 for i in x], val_after_values, width=bar_width, label='Sesudah', color='salmon')
plt.xticks(x, val_labels, rotation=45, ha='right')
plt.ylabel("Jumlah Citra")
plt.title("Perbandingan Jumlah Citra Validation per Kelas (Sebelum vs Sesudah)")
plt.legend()
plt.grid(axis='y', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

# =======================
# Plot Perbandingan Total
# =======================

# Hitung total gambar sebelum proses
train_counts_before = count_images_per_class(before_train_dir)
val_counts_before = count_images_per_class(before_val_dir)

total_before_train = sum(train_counts_before.values())
total_before_val = sum(val_counts_before.values())
total_after_train = sum(train_counts_after.values())
total_after_val = sum(val_counts_after.values())

# Plot perbandingan total
labels = ['Train Sebelum', 'Train Sesudah', 'Val Sebelum', 'Val Sesudah']
values = [total_before_train, total_after_train, total_before_val, total_after_val]
colors = ['gray', 'skyblue', 'gray', 'salmon']

plt.figure(figsize=(10, 6))
plt.bar(labels, values, color=colors)
plt.title("Perbandingan Total Data Citra Sebelum dan Sesudah Proses Alignment & Augmentasi")
plt.ylabel("Jumlah Total Citra")
plt.grid(axis='y', linestyle='--', linewidth=0.7, alpha=0.7)
plt.tight_layout()
plt.show()

# ========== Konfigurasi ========== #
train_dir = "D:/Kuliah/Skripsi/Data_StratifiedSplit/train"
val_dir = "D:/Kuliah/Skripsi/Data_StratifiedSplit/val"
num_samples = 5  # jumlah gambar per set yang ditampilkan

# ========== Fungsi untuk menampilkan contoh gambar ========== #
def show_sample_images(data_dir, title):
    class_names = os.listdir(data_dir)
    class_names = [cls for cls in class_names if os.path.isdir(os.path.join(data_dir, cls))]

    sampled_images = []

    for cls in class_names:
        cls_folder = os.path.join(data_dir, cls)
        images = os.listdir(cls_folder)
        if images:
            sample = random.choice(images)
            sampled_images.append((os.path.join(cls_folder, sample), cls))
        if len(sampled_images) >= num_samples:
            break

    # Plot gambar
    plt.figure(figsize=(15, 3))
    for i, (img_path, label) in enumerate(sampled_images):
        img = Image.open(img_path)
        plt.subplot(1, num_samples, i + 1)
        plt.imshow(img)
        plt.title(f"Label: {label}")
        plt.axis("off")
    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

# ========== Menampilkan contoh dari train dan val ========== #
show_sample_images(train_dir, "Contoh Gambar dari Training Set")
show_sample_images(val_dir, "Contoh Gambar dari Validation Set")

MODEL EFFICIENTNETV2-SMALL WITHOUT PRE-TRAINED

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ====== 1. Setup device ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ====== 2. Transformasi dan Dataset ======
mean, std = [0.5]*3, [0.5]*3
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

data_dir = "D:/Kuliah/Skripsi/Data_Preprocessed"
train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=transform)
val_dataset   = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=transform)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir, "val1"), transform=transform)  # ✅ Data uji tambahan

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)       # ✅

class_names = train_dataset.classes
num_classes = len(class_names)

# ====== 3. Load EfficientNetV2-S tanpa pretrained ======
from torchvision.models import efficientnet_v2_s
model = efficientnet_v2_s(weights=None)
model.classifier = nn.Linear(1280, num_classes)
model = model.to(device)

# ====== 4. Loss, Optimizer, Scheduler ======
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# ====== 5. Training Loop ======
epochs = 20
train_losses = []
train_accuracies = []
val_accuracies = []

for epoch in range(epochs):
    model.train()
    train_loss, train_correct = 0.0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * imgs.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += torch.sum(preds == labels).item()

    scheduler.step()

    train_loss /= len(train_loader.dataset)
    train_acc = train_correct / len(train_loader.dataset)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)

    # ====== Validation ======
    model.eval()
    val_correct, y_true_val, y_pred_val = 0, [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            val_correct += torch.sum(preds == labels).item()
            y_true_val.extend(labels.cpu().numpy())
            y_pred_val.extend(preds.cpu().numpy())

    val_acc = val_correct / len(val_loader.dataset)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# ====== 6. Simpan Model ======
torch.save(model.state_dict(), "efficientnetv2s_jkt48_from_scratch1.pth")
print("\n✅ Model berhasil disimpan ke 'efficientnetv2s_jkt48_from_scratch1.pth'")

# ====== 7. Evaluasi Data Test ======
print("\n📦 Evaluasi pada data test...")

model.eval()
y_true_test, y_pred_test = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        _, preds = torch.max(outputs, 1)
        y_true_test.extend(labels.cpu().numpy())
        y_pred_test.extend(preds.cpu().numpy())

# ====== 8. Hasil Evaluasi Test ======
print("\nClassification Report (Test):")
print(classification_report(y_true_test, y_pred_test, target_names=class_names))

cm_test = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_test, 
            xticklabels=class_names, 
            yticklabels=class_names, 
            cmap="Blues", 
            fmt="d", 
            annot=True, 
            annot_kws={"size": 8})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - EfficientNetV2S")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ====== 9. Plot Train Loss & Val Accuracy ======
best_val_acc = max(val_accuracies)
best_val_epoch = val_accuracies.index(best_val_acc) + 1

best_train_loss = min(train_losses)
best_loss_epoch = train_losses.index(best_train_loss) + 1

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), train_losses, marker='o', label='Train Loss')
plt.scatter(best_loss_epoch, best_train_loss, color='red', label=f'Best Loss (Epoch {best_loss_epoch})')
plt.text(best_loss_epoch, best_train_loss + 0.02, f"{best_train_loss:.4f}", color='red', ha='center')
plt.title("Train Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()

plt.subplot(1, 2, 2)
epochs_range = range(1, epochs + 1)
plt.plot(epochs_range, train_accuracies, marker='o', label='Train Accuracy', color='orange')
plt.plot(epochs_range, val_accuracies, marker='o', label='Validation Accuracy', color='green')
plt.scatter(best_val_epoch, best_val_acc, color='blue', label=f'Best Val Acc (Epoch {best_val_epoch})')
plt.text(best_val_epoch, best_val_acc + 0.01, f"{best_val_acc:.4f}", color='blue', ha='center')
plt.title("Train & Validation Accuracy per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()


PCA AND SVM

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib
from torchvision import transforms
from torchvision.models import efficientnet_v2_s
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ====== 1. Setup ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset kustom agar mengembalikan path
class ImageFolderWithPaths(ImageFolder):
    def __getitem__(self, index):
        original_tuple = super().__getitem__(index)
        path = self.imgs[index][0]
        return original_tuple + (path,)

# Transformasi dan loader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

train_dataset = ImageFolderWithPaths("D:/Kuliah/Skripsi/Data_Preprocessed/train", transform=transform)
val_dataset = ImageFolderWithPaths("D:/Kuliah/Skripsi/Data_Preprocessed/val1", transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

class_names = train_dataset.classes

# Load EfficientNetV2-S sebagai feature extractor
model = efficientnet_v2_s(weights=None)
model.classifier = nn.Linear(1280, 40)
model.load_state_dict(torch.load("efficientnetv2s_jkt48_from_scratch1.pth"))
model.classifier = nn.Identity()
model = model.to(device)
model.eval()

# ====== 2. Ekstraksi Fitur ======
def extract_features(dataloader):
    features, labels_all, paths_all = [], [], []
    with torch.no_grad():
        for imgs, labels, paths in dataloader:
            imgs = imgs.to(device)
            feats = model(imgs)
            features.append(feats.cpu().numpy())
            labels_all.extend(labels.numpy())
            paths_all.extend(paths)
    return np.concatenate(features), np.array(labels_all), paths_all

print("\n🔍 Mengekstrak fitur dari data train dan val...")
X_train, y_train, path_train = extract_features(train_loader)
X_val, y_val, path_val = extract_features(val_loader)
print("✅ Fitur diekstraksi:", X_train.shape)

# ====== 3. Simpan Fitur Mentah ======
df_train_feat = pd.DataFrame(X_train)
df_train_feat["label"] = y_train
df_train_feat["path"] = path_train

df_val_feat = pd.DataFrame(X_val)
df_val_feat["label"] = y_val
df_val_feat["path"] = path_val

with pd.ExcelWriter("fitur_efficientnetv2s.xlsx") as writer:
    df_train_feat.to_excel(writer, sheet_name="Train_Fitur", index=False)
    df_val_feat.to_excel(writer, sheet_name="Val_Fitur", index=False)

# ====== 4. PCA ======
print("\n⚙️  Melakukan PCA...")
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

# Plot cumulative explained variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-')
plt.axhline(y=0.95, color='red', linestyle='--', label='95% Variance Threshold')
plt.title('Cumulative Explained Variance by PCA Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Simpan hasil PCA
df_train_pca = pd.DataFrame(X_train_pca)
df_train_pca["label"] = y_train
df_train_pca["path"] = path_train

df_val_pca = pd.DataFrame(X_val_pca)
df_val_pca["label"] = y_val
df_val_pca["path"] = path_val

df_var = pd.DataFrame({
    "Component": np.arange(1, len(cumulative_variance) + 1),
    "ExplainedVariance": pca.explained_variance_ratio_,
    "Cumulative": cumulative_variance
})

with pd.ExcelWriter("fitur_pca_efficientnetv2s.xlsx") as writer:
    df_train_pca.to_excel(writer, sheet_name="Train_PCA", index=False)
    df_val_pca.to_excel(writer, sheet_name="Val_PCA", index=False)
    df_var.to_excel(writer, sheet_name="PCA_Variance", index=False)

# ====== 5. Training SVM ======
print("\n🔧 Training SVM classifier...")
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train_pca, y_train)

# Simpan model SVM
joblib.dump(svm, "svm_pca_efficientnetv2s.pkl")
print("✅ Model SVM berhasil disimpan ke 'svm_pca_efficientnetv2s.pkl'")

# Simpan model feature extractor
torch.save(model.state_dict(), "efficientnetv2s_feature_extractor.pth")
print("✅ Model feature extractor berhasil disimpan ke 'efficientnetv2s_feature_extractor.pth'")

# ====== 6. Evaluasi ======
y_pred_svm = svm.predict(X_val_pca)
print("\n🎯 Accuracy PCA+SVM:", accuracy_score(y_val, y_pred_svm))
print("\nClassification Report (PCA + SVM):")
print(classification_report(y_val, y_pred_svm, target_names=class_names))

# ====== 7. Confusion Matrix ======
cm = confusion_matrix(y_val, y_pred_svm)
plt.figure(figsize=(12, 10))
sns.heatmap(cm,
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues",
            fmt="d",
            annot=True,
            annot_kws={"size": 8})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - PCA + SVM")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


HYPERPARAMETER TUNING

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.base import clone

# ====== 1. Load fitur hasil ekstraksi dari Excel ======
df_train_feat = pd.read_excel("fitur_efficientnetv2s.xlsx", sheet_name="Train_Fitur")
df_val_feat = pd.read_excel("fitur_efficientnetv2s.xlsx", sheet_name="Val_Fitur")

X_train = df_train_feat.drop(columns=["label"]).values
y_train = df_train_feat["label"].values
X_val = df_val_feat.drop(columns=["label"]).values
y_val = df_val_feat["label"].values

# ====== 2. Pipeline PCA + SVM ======
pipe = Pipeline([
    ('pca', PCA()),
    ('svm', SVC())
])

# ====== 3. Parameter Space ======
param_distributions = {
    'pca__n_components': [0.90, 0.95, 0.99],
    'svm__C': [0.1, 1, 10],
    'svm__gamma': ['scale', 0.001, 0.01, 0.1, 1]
}

# ====== 4. Randomized Search ======
search = RandomizedSearchCV(
    pipe,
    param_distributions=param_distributions,
    n_iter=50,
    scoring='accuracy',
    cv=3,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, y_train)
best_model = search.best_estimator_

# ====== Tampilkan Best Params ======
print("\n===== BEST PARAMETERS FOUND =====")
print(search.best_params_)
print(f"Cross-Validation Accuracy: {search.best_score_:.4f}")

from sklearn.metrics import f1_score
# ====== 5. Evaluasi hasil tuning: CV + Validasi + Inference Time + F1-Score ======
cv_results = pd.DataFrame(search.cv_results_).sort_values(by='mean_test_score', ascending=False)

combined_results = []
param_list = search.cv_results_['params']
cv_scores = search.cv_results_['mean_test_score']

for i, (params, cv_score) in enumerate(zip(param_list, cv_scores)):
    print(f"🔁 Evaluasi dan ukur waktu model ke-{i+1}")
    model = clone(pipe)
    model.set_params(**params)
    model.fit(X_train, y_train)

    # Hitung waktu inferensi
    start_time = time.time()
    y_val_pred = model.predict(X_val)
    end_time = time.time()
    inference_time = end_time - start_time

    val_acc = accuracy_score(y_val, y_val_pred)
    val_f1_macro = f1_score(y_val, y_val_pred, average='macro')

    combined_results.append({
        'pca__n_components': params['pca__n_components'],
        'svm__C': params['svm__C'],
        'svm__gamma': params['svm__gamma'],
        'cv_accuracy': cv_score,
        'val_accuracy': val_acc,
        'val_f1_macro': val_f1_macro,
        'inference_time_seconds': inference_time
    })

compare_df = pd.DataFrame(combined_results).sort_values(by='val_accuracy', ascending=False)


# ====== 6. Evaluasi akhir dari model terbaik ======
y_pred_val = best_model.predict(X_val)
val_acc = accuracy_score(y_val, y_pred_val)
val_report_dict = classification_report(y_val, y_pred_val, output_dict=True)
val_report_df = pd.DataFrame(val_report_dict).transpose()
val_summary = pd.DataFrame({
    "Metric": ["Validation Accuracy"],
    "Value": [val_acc]
})

# ====== 7. Simpan semua hasil ke Excel ======
with pd.ExcelWriter("hasil_tuning_pca_svm.xlsx") as writer:
    cv_results.to_excel(writer, sheet_name="Tuning_Results_CV", index=False)
    compare_df.to_excel(writer, sheet_name="CV_vs_Val_Accuracy", index=False)
    val_summary.to_excel(writer, sheet_name="Val_Evaluation", index=False)
    val_report_df.to_excel(writer, sheet_name="Classification_Report")

print("✅ Semua hasil disimpan ke 'hasil_tuning_pca_svm2.xlsx'")

# ====== 9. Visualisasi Waktu Inferensi ======
plt.figure(figsize=(10, 6))
plt.plot(compare_df['inference_time_seconds'].values, marker='o', label="Inference Time (s)", color='orange')
plt.title("Waktu Inferensi Setiap Model (Validation Set)")
plt.xlabel("Model Ke-i")
plt.ylabel("Inference Time (detik)")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


from sklearn.preprocessing import LabelEncoder

# ====== Encode label dan simpan label encoder ======
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

# Refit best_model dengan y_train_encoded
best_model.fit(X_train, y_train_encoded)

# Simpan pipeline + label encoder ke dict
pipeline_dict = {
    "pipeline": best_model,
    "label_encoder": label_encoder
}

joblib.dump(pipeline_dict, "best_pca_svm_pipeline_with_encoder.pkl")
print("✅ Pipeline + Label Encoder disimpan ke 'pca_svm_pipeline_with_encoder.pkl'")

UJI COBA MODEL TERBAIK

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from pathlib import Path

# ====== 1. Load Class Names (From Folder Names) ======
train_dir = Path("D:/Kuliah/Skripsi/Data_Preprocessed/train")
class_names = sorted([folder.name for folder in train_dir.iterdir() if folder.is_dir()])

# Buat mapping ID (angka) ke Nama Kelas (folder)
id_to_classname = dict(enumerate(class_names))

# ====== 2. Load Fitur (Train & Val) ======
df_train_feat = pd.read_excel("fitur_efficientnetv2s.xlsx", sheet_name="Train_Fitur")
df_val_feat = pd.read_excel("fitur_efficientnetv2s.xlsx", sheet_name="Val_Fitur")

# ====== 3. Mapping Label ID -> Nama Folder ======
df_train_feat["label"] = df_train_feat["label"].map(id_to_classname)
df_val_feat["label"] = df_val_feat["label"].map(id_to_classname)

# Pastikan label bertipe string
df_train_feat["label"] = df_train_feat["label"].astype(str)
df_val_feat["label"] = df_val_feat["label"].astype(str)

# ====== 4. Siapkan X & y ======
X_train = df_train_feat.drop(columns=["label"]).values
y_train = df_train_feat["label"].values
X_val = df_val_feat.drop(columns=["label"]).values
y_val = df_val_feat["label"].values

# ====== 5. Label Encoding (Sesuai Nama Folder) ======
label_encoder = LabelEncoder()
label_encoder.fit(class_names)

y_train_enc = label_encoder.transform(y_train)
y_val_enc = label_encoder.transform(y_val)

# ====== 6. PCA ======
print("\n⚙️  Melakukan PCA...")
pca = PCA(n_components=0.99)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

# Plot cumulative explained variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(cumulative_variance) + 1), cumulative_variance, marker='o', linestyle='-')
plt.axhline(y=0.99, color='red', linestyle='--', label='99% Variance Threshold')
plt.title('Cumulative Explained Variance by PCA Components')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# ====== 7. Train SVM ======
print("\n🔧 Training SVM classifier...")
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train_pca, y_train_enc)

# Simpan PCA + SVM + Label Encoder ke dalam 1 file
pipeline = {
    "pca": pca,
    "svm": svm,
    "label_encoder": label_encoder
}
joblib.dump(pipeline, "best_pca_svm_pipeline_efficientnetv2s-1.pkl")
print("✅ PCA, SVM, dan Label Encoder disimpan ke 'best_pca_svm_pipeline_efficientnetv2s-1.pkl'")

# ====== 8. Evaluasi ======
y_pred_enc = svm.predict(X_val_pca)
y_pred_labels = label_encoder.inverse_transform(y_pred_enc)
y_true_labels = label_encoder.inverse_transform(y_val_enc)

# Evaluasi Classification Report
print("\n🎯 Accuracy PCA+SVM:", accuracy_score(y_true_labels, y_pred_labels))
print("\nClassification Report (PCA + SVM):")
print(classification_report(y_true_labels, y_pred_labels, target_names=class_names))

# ====== 9. Confusion Matrix ======
cm = confusion_matrix(y_true_labels, y_pred_labels, labels=class_names)

plt.figure(figsize=(12, 10))
sns.heatmap(cm,
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues",
            fmt="d",
            annot=True,
            annot_kws={"size": 8})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - PCA + SVM (Terbaik)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

EffNetV2S tanpa PCA

In [ ]:
import pandas as pd
import joblib
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ========== 1. Load Fitur dari File ==========
print("\n📥 Memuat fitur dari file Excel...")
df_train_feat = pd.read_excel("fitur_efficientnetv2s.xlsx", sheet_name="Train_Fitur")
df_val_feat = pd.read_excel("fitur_efficientnetv2s.xlsx", sheet_name="Val_Fitur")

X_train = df_train_feat.drop(columns=["label"]).values
y_train = df_train_feat["label"].values
X_val = df_val_feat.drop(columns=["label"]).values
y_val = df_val_feat["label"].values

print("✅ Data dimuat:")
print(f"  - X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  - X_val  : {X_val.shape}, y_val  : {y_val.shape}")

# ========== 2. Training SVM ==========
print("\n🔧 Training SVM classifier...")
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train, y_train)

# Simpan model SVM
joblib.dump(svm, "svm_efficientnetv2s_tanpa_pca.pkl")
print("✅ Model SVM berhasil disimpan ke 'svm_efficientnetv2s_tanpa_pca.pkl'")

# ========== 3. Evaluasi ==========
y_pred_svm = svm.predict(X_val)
print("\n🎯 Accuracy EfficientNetV2-S + SVM (tanpa PCA):", accuracy_score(y_val, y_pred_svm))

# Ganti dengan daftar nama kelas yang kamu punya
class_names = [f"Kelas {i}" for i in range(40)]

print("\nClassification Report (tanpa PCA):")
print(classification_report(y_val, y_pred_svm, target_names=class_names))

# ========== 4. Confusion Matrix ==========
cm = confusion_matrix(y_val, y_pred_svm)

plt.figure(figsize=(12, 10))
sns.heatmap(cm,
            xticklabels=class_names,
            yticklabels=class_names,
            cmap="Blues",
            fmt="d",
            annot=True,
            annot_kws={"size": 8})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - EfficientNetV2S + SVM (tanpa PCA)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

EFFICIENTNETV2-SMALL PRETRAINED

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# ====== 1. Pengaturan Device ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ====== 2. Preprocessing dan Augmentasi ======
mean, std = [0.5]*3, [0.5]*3

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# ====== 3. Load Dataset dari Folder ======
data_dir = "D:/Kuliah/Skripsi/Data_Preprocessed"
train_dir = os.path.join(data_dir, "train")
val_dir = os.path.join(data_dir, "val")

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

num_classes = len(train_dataset.classes)
print("Jumlah kelas:", num_classes)

# ====== 4. Model EfficientNetV2-S ======
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
model.classifier = nn.Linear(1280, num_classes)
model = model.to(device)

# ====== 5. Optimizer, Loss, Scheduler ======
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

# ====== 6. Training Loop ======
epochs = 20
for epoch in range(epochs):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    acc = correct / total
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss:.4f}, Accuracy: {acc:.4f}")
    scheduler.step()

# ====== 7. Simpan Model ======
save_path = "efficientnetv2s_finetuned_jkt48.pth"
torch.save(model, save_path)
print(f"\n✅ Model berhasil disimpan di: {save_path}")

# ====== 8. Evaluasi pada Data Uji Eksternal ======
test_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val1"  
test_dataset = datasets.ImageFolder(test_dir, transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap="Blues", annot=True, fmt="d", cbar=False,
            xticklabels=test_dataset.classes, yticklabels=test_dataset.classes,
            annot_kws={"size": 10})
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix ")
plt.tight_layout()
plt.show()

# Classification Report
report = classification_report(y_true, y_pred, target_names=test_dataset.classes)
print("\nClassification Report :\n")
print(report)


EFFICIENTNETV2-SMALL PRETRAINED+PCA+SVM

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import joblib
import pandas as pd
from tqdm import tqdm

# ===== 1. Konfigurasi Path dan Device =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
data_dir = "D:/Kuliah/Skripsi/Data_Preprocessed"
train_dir = os.path.join(data_dir, "train")
val_dir = os.path.join(data_dir, "val")
test_dir = os.path.join(data_dir, "val1")  # Folder data uji baru

extractor_model_path = "efficientnet_extractor_finetuned.pth"
pca_svm_model_path = "pca_svm_pipeline_finetuned.pkl"
label_encoder_path = "label_encoder_finetuned.pkl"
hasil_excel_path = "hasil_klasifikasi_finetuned.xlsx"
hasil_excel_test_path = "hasil_klasifikasi_test.xlsx"  # output uji data lain

# ===== 2. Transformasi Data =====
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder(train_dir, transform=transform)
val_dataset = datasets.ImageFolder(val_dir, transform=transform)
test_dataset = datasets.ImageFolder(test_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# ===== 3. Load dan Konfigurasi Model =====
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
model = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
model.classifier = nn.Identity()
model = model.to(device)
model.eval()

# ===== 4. Fungsi Ekstraksi Fitur =====
def extract_features(dataloader):
    features, labels = [], []
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, desc="Extracting Features"):
            imgs = imgs.to(device)
            out = model(imgs).cpu().numpy()
            features.append(out)
            labels.extend(lbls.numpy())
    return np.vstack(features), np.array(labels)

# ===== 5. Ekstraksi Fitur untuk Train dan Val =====
X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)

# ===== 6. Simpan Model Ekstraktor (Optional) =====
torch.save(model.state_dict(), extractor_model_path)

# ===== 7. Label Encoding =====
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
joblib.dump(label_encoder, label_encoder_path)

# ===== 8. PCA + SVM Training =====
pca = PCA(n_components=0.95, svd_solver='full')
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train_pca, y_train_enc)

# ===== 9. Simpan Pipeline PCA + SVM =====
pipeline = {"pca": pca, "svm": svm}
joblib.dump(pipeline, pca_svm_model_path)

# ===== 10. Evaluasi pada Data Validasi =====
y_pred_val_enc = svm.predict(X_val_pca)
y_pred_val_labels = label_encoder.inverse_transform(y_pred_val_enc)
y_val_labels = label_encoder.inverse_transform(y_val_enc)

report_val = classification_report(y_val_labels, y_pred_val_labels, output_dict=True)
df_val = pd.DataFrame(report_val).transpose()
df_val.to_excel(hasil_excel_path)
print("\nClassification Report (Validation):\n", df_val)

# Confusion Matrix untuk Validasi
cm_val = confusion_matrix(y_val_labels, y_pred_val_labels, labels=val_dataset.classes)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_val, annot=True, fmt="d", cmap="Blues",
            xticklabels=val_dataset.classes,
            yticklabels=val_dataset.classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

# ===== 11. Evaluasi pada Data Uji (Data Lain) =====
X_test, y_test = extract_features(test_loader)
y_test_enc = label_encoder.transform(y_test)  # transform label sesuai label_encoder

X_test_pca = pca.transform(X_test)
y_pred_test_enc = svm.predict(X_test_pca)
y_pred_test_labels = label_encoder.inverse_transform(y_pred_test_enc)
y_test_labels = label_encoder.inverse_transform(y_test_enc)

# Classification Report - Test
report_test = classification_report(y_test_labels, y_pred_test_labels, output_dict=True)
df_test = pd.DataFrame(report_test).transpose()
df_test.to_excel(hasil_excel_test_path)
print("\nClassification Report :\n", df_test)

# Confusion Matrix untuk Test
cm_test = confusion_matrix(y_test_labels, y_pred_test_labels, labels=test_dataset.classes)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues",
            xticklabels=test_dataset.classes,
            yticklabels=test_dataset.classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()


EIGENFACES

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import joblib

# ====== 1. Setup Path & Ukuran Gambar ======
IMAGE_SIZE = (128, 128)
train_dir = Path("D:/Kuliah/Skripsi/Data_Preprocessed/train")
val_dir = Path("D:/Kuliah/Skripsi/Data_Preprocessed/val")
test_dir = Path("D:/Kuliah/Skripsi/Data_Preprocessed/val1")  # Tambahan: direktori data uji baru

# ====== 2. Fungsi Load Gambar dan Label ======
def load_images_labels(directory):
    images, labels = [], []
    for class_dir in sorted(directory.iterdir()):
        if class_dir.is_dir():
            for img_path in class_dir.glob("*.jpg"):
                img = load_img(img_path, target_size=IMAGE_SIZE)
                img_array = img_to_array(img).astype("uint8")
                img_gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
                images.append(img_gray.flatten())
                labels.append(class_dir.name)
    return np.array(images), np.array(labels)

# ====== 3. Load Dataset ======
X_train, y_train = load_images_labels(train_dir)
X_val, y_val = load_images_labels(val_dir)
X_test, y_test = load_images_labels(test_dir)

# ====== 4. Encode Label ======
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)
class_names = label_encoder.classes_

# ====== 5. PCA ======
pca = PCA(n_components=0.95, whiten=True)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)
X_test_pca = pca.transform(X_test)  # Data uji juga diproyeksikan ke ruang PCA

# ====== 6. SVM Training ======
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train_pca, y_train_encoded)

# ====== 7. Evaluasi di Data Test Baru ======
y_pred_test = svm.predict(X_test_pca)
acc_test = accuracy_score(y_test_encoded, y_pred_test)
print(f"\n🎯 Akurasi pada data uji (test): {acc_test:.4f}")

# Classification Report & Save
report_test = classification_report(y_test_encoded, y_pred_test, target_names=class_names, output_dict=True)
pd.DataFrame(report_test).transpose().to_excel("classification_report_test_eigenfaces.xlsx")
print("📄 Classification report (test) disimpan ke 'classification_report_test_eigenfaces.xlsx'")

# Confusion Matrix & Save
cm_test = confusion_matrix(y_test_encoded, y_pred_test)
cm_df_test = pd.DataFrame(cm_test, index=class_names, columns=class_names)
cm_df_test.to_excel("confusion_matrix_test_eigenfaces.xlsx")
print("📄 Confusion matrix (test) disimpan ke 'confusion_matrix_test_eigenfaces.xlsx'")

# ====== 8. Simpan Model ======
joblib.dump(pca, "model_pca_eigenfaces.pkl")
joblib.dump(svm, "model_svm_eigenfaces.pkl")
joblib.dump(label_encoder, "label_encoder.pkl")
print("✅ Model PCA, SVM, dan LabelEncoder disimpan")

# ====== 9. Visualisasi Confusion Matrix (Test) ======
plt.figure(figsize=(12, 10))
sns.heatmap(cm_df_test, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix - Test Data (Eigenfaces + SVM)")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


FISHERFACES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import cv2
from tensorflow.keras.utils import load_img, img_to_array

# ========== 1. Load gambar grayscale dan label dari folder ==========
IMAGE_SIZE = (128, 128)  # Resolusi gambar

def load_images_labels(directory):
    images, labels = [], []
    for class_dir in sorted(directory.iterdir()):
        if class_dir.is_dir():
            for img_path in class_dir.glob("*.jpg"):
                img = load_img(img_path, target_size=IMAGE_SIZE)
                img_array = img_to_array(img).astype("uint8")
                img_gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
                images.append(img_gray.flatten())
                labels.append(class_dir.name)
    return np.array(images), np.array(labels)

# Folder dataset
train_dir = Path("D:/Kuliah/Skripsi/Data_Preprocessed/train")
val_dir = Path("D:/Kuliah/Skripsi/Data_Preprocessed/val")
class_names = sorted([d.name for d in train_dir.iterdir() if d.is_dir()])

# Load train dan val
X_train, y_train = load_images_labels(train_dir)
X_val, y_val = load_images_labels(val_dir)

# ========== 2. PCA untuk mereduksi dimensi sebelum LDA ==========
pca = PCA(n_components=0.95, svd_solver='full', whiten=True)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

# ========== 3. LDA (Fisherfaces) ==========
lda = LDA()
X_train_lda = lda.fit_transform(X_train_pca, y_train)
X_val_lda = lda.transform(X_val_pca)

# ========== 4. Klasifikasi dengan SVM ==========
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train_lda, y_train)

# ========== 5. Evaluasi ==========
y_pred = svm.predict(X_val_lda)

print("\n📋 Classification Report:")
print(classification_report(y_val, y_pred, target_names=class_names))

# Confusion Matrix
cm = confusion_matrix(y_val, y_pred, labels=class_names)

plt.figure(figsize=(12, 10))
sns.heatmap(cm, 
            xticklabels=class_names, 
            yticklabels=class_names, 
            cmap="Blues", 
            fmt="d", 
            annot=True, 
            annot_kws={"size": 8})
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Fisherfaces (LDA + PCA)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# ========== 6. Simpan Model ==========
joblib.dump({'pca': pca, 'lda': lda, 'svm': svm}, "fisherfaces_model.pkl")
print("✅ Model Fisherfaces disimpan sebagai 'fisherfaces_model.pkl'")

# ====== 7. Simpan hasil evaluasi ke Excel ======
from pandas import ExcelWriter

# 1. Classification report ke DataFrame
report_dict = classification_report(y_val, y_pred, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report_dict).transpose()

# 2. Confusion matrix ke DataFrame
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)

# 3. Ringkasan akurasi
summary_df = pd.DataFrame({
    "Metric": ["Validation Accuracy"],
    "Value": [svm.score(X_val_lda, y_val)]
})

# ========== 8. Uji Model pada Data Baru (Test) ==========
test_dir = Path("D:/Kuliah/Skripsi/Data_Preprocessed/val1")  # Ganti dengan path data test-mu

if test_dir.exists():
    X_test, y_test = load_images_labels(test_dir)
    X_test_pca = pca.transform(X_test)
    X_test_lda = lda.transform(X_test_pca)
    y_test_pred = svm.predict(X_test_lda)

    print("\n📋 Classification Report (Test Data):")
    print(classification_report(y_test, y_test_pred, target_names=class_names))

    # Confusion Matrix (Test)
    cm_test = confusion_matrix(y_test, y_test_pred, labels=class_names)

    plt.figure(figsize=(12, 10))
    sns.heatmap(cm_test,
                xticklabels=class_names,
                yticklabels=class_names,
                cmap="Oranges",
                fmt="d",
                annot=True,
                annot_kws={"size": 8})
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title("Confusion Matrix - Test Data")
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

    # Simpan hasil evaluasi test ke Excel
    report_test_df = pd.DataFrame(classification_report(y_test, y_test_pred, target_names=class_names, output_dict=True)).transpose()
    cm_test_df = pd.DataFrame(cm_test, index=class_names, columns=class_names)
    summary_test_df = pd.DataFrame({
        "Metric": ["Test Accuracy"],
        "Value": [svm.score(X_test_lda, y_test)]
    })

    with ExcelWriter("hasil_fisherfaces.xlsx", mode="a", engine="openpyxl") as writer:
        report_test_df.to_excel(writer, sheet_name="Classification_Report_Test")
        cm_test_df.to_excel(writer, sheet_name="Confusion_Matrix_Test")
        summary_test_df.to_excel(writer, sheet_name="Accuracy_Summary_Test", index=False)

ALEXNET

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ====== Konfigurasi ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/train"
val_dir   = "D:/Kuliah/Skripsi/Data_Preprocessed/val"
batch_size = 32
num_epochs = 20
model_save_path = "alexnet_model.pth"
excel_save_path = "hasil_klasifikasi_alexnet.xlsx"

# ====== Load Class Names ======
class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)

# ====== Transformasi Data ======
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
}

# ====== Load Dataset ======
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms['train'])
val_dataset   = datasets.ImageFolder(val_dir, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ====== Load AlexNet ======
model = models.alexnet(pretrained=True)
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
model = model.to(device)

# ====== Loss dan Optimizer ======
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ====== Training ======
print("[INFO] Training started...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

# ====== Simpan Model ======
torch.save(model.state_dict(), model_save_path)
print(f"[INFO] Model disimpan di: {model_save_path}")

# ====== Evaluasi ======
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

# ====== Classification Report ======
print("\n[INFO] Classification Report:")
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print(classification_report(y_true, y_pred, target_names=class_names))

# ====== Confusion Matrix ======
cm = confusion_matrix(y_true, y_pred)

# ====== Simpan ke Excel ======
with pd.ExcelWriter(excel_save_path) as writer:
    # Classification Report
    df_report = pd.DataFrame(report).transpose()
    df_report.to_excel(writer, sheet_name='Classification Report')

    # Confusion Matrix
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    df_cm.to_excel(writer, sheet_name='Confusion Matrix')

print(f"[INFO] Hasil evaluasi disimpan di: {excel_save_path}")

# ====== Plot Confusion Matrix ======
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - AlexNet")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ====== Pengujian Data Baru (Test Dir) ======
test_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val1" 
if os.path.exists(test_dir):
    print("\n[INFO] Pengujian data baru dimulai...")

    test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms['val'])
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    test_class_names = test_dataset.classes  # bisa berbeda jika struktur folder test berbeda

    y_true_test = []
    y_pred_test = []

    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true_test.extend(labels.numpy())
            y_pred_test.extend(preds.cpu().numpy())

    # Classification Report - Test
    print("\n[INFO] Classification Report (Data Baru):")
    test_report = classification_report(y_true_test, y_pred_test, target_names=test_class_names, output_dict=True)
    print(classification_report(y_true_test, y_pred_test, target_names=test_class_names))

    # Confusion Matrix - Test
    cm_test = confusion_matrix(y_true_test, y_pred_test)

    # Simpan ke Excel
    with pd.ExcelWriter(excel_save_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_test_report = pd.DataFrame(test_report).transpose()
        df_test_report.to_excel(writer, sheet_name='Test Report')

        df_test_cm = pd.DataFrame(cm_test, index=test_class_names, columns=test_class_names)
        df_test_cm.to_excel(writer, sheet_name='Test Confusion Matrix')

    print(f"[INFO] Hasil pengujian data baru disimpan di: {excel_save_path}")

    # Plot Confusion Matrix - Test
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm_test, annot=True, fmt='d', xticklabels=test_class_names, yticklabels=test_class_names, cmap='YlGnBu')
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix - Data Baru (Test)")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print("\n[WARNING] Folder test_dir tidak ditemukan. Lewati pengujian data baru.")


RESNET18

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ====== Konfigurasi ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/train"
val_dir   = "D:/Kuliah/Skripsi/Data_Preprocessed/val"
batch_size = 32
num_epochs = 20
model_save_path = "resnet18_model.pth"
excel_save_path = "hasil_klasifikasi_resnet18.xlsx"

# ====== Load Class Names ======
class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)

# ====== Transformasi Data ======
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
}

# ====== Load Dataset ======
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms['train'])
val_dataset   = datasets.ImageFolder(val_dir, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ====== Load ResNet18 ======
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# ====== Loss dan Optimizer ======
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ====== Training ======
print("[INFO] Training started...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

# ====== Simpan Model ======
torch.save(model.state_dict(), model_save_path)
print(f"[INFO] Model disimpan di: {model_save_path}")

# ====== Evaluasi ======
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

# ====== Classification Report ======
print("\n[INFO] Classification Report:")
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print(classification_report(y_true, y_pred, target_names=class_names))

# ====== Confusion Matrix ======
cm = confusion_matrix(y_true, y_pred)

# ====== Simpan ke Excel ======
with pd.ExcelWriter(excel_save_path) as writer:
    # Classification Report
    df_report = pd.DataFrame(report).transpose()
    df_report.to_excel(writer, sheet_name='Classification Report')

    # Confusion Matrix
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    df_cm.to_excel(writer, sheet_name='Confusion Matrix')

print(f"[INFO] Hasil evaluasi disimpan di: {excel_save_path}")

# ====== Plot Confusion Matrix ======
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - ResNet18")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ====== Pengujian Data Baru ======
test_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val1" 

if os.path.exists(test_dir):
    print("\n[INFO] Menguji data baru dari folder:", test_dir)

    test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms['val'])
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    y_true_test = []
    y_pred_test = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true_test.extend(labels.numpy())
            y_pred_test.extend(preds.cpu().numpy())

    # Classification report dan confusion matrix
    print("\n[INFO] Classification Report (Test Data):")
    report_test = classification_report(
        y_true_test, y_pred_test, target_names=class_names, output_dict=True
    )
    print(classification_report(y_true_test, y_pred_test, target_names=class_names))

    cm_test = confusion_matrix(y_true_test, y_pred_test)

    # Simpan ke Excel
    with pd.ExcelWriter(excel_save_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_report_test = pd.DataFrame(report_test).transpose()
        df_report_test.to_excel(writer, sheet_name='Test Report')

        df_cm_test = pd.DataFrame(cm_test, index=class_names, columns=class_names)
        df_cm_test.to_excel(writer, sheet_name='Test Confusion Matrix')

    print(f"[INFO] Hasil pengujian test data disimpan di: {excel_save_path}")

    # Plot confusion matrix
    plt.figure(figsize=(14, 12))
    sns.heatmap(cm_test, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Oranges')
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix - Test Data")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

else:
    print(f"[WARNING] Folder test_dir tidak ditemukan: {test_dir}")


VGGNET16

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# ====== Konfigurasi ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
train_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/train"
val_dir   = "D:/Kuliah/Skripsi/Data_Preprocessed/val"
batch_size = 32
num_epochs = 10
model_save_path = "vgg16_model.pth"
excel_save_path = "hasil_klasifikasi_vgg16.xlsx"

# ====== Load Class Names ======
class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)

# ====== Transformasi Data ======
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5])
    ]),
}

# ====== Load Dataset ======
train_dataset = datasets.ImageFolder(train_dir, transform=data_transforms['train'])
val_dataset   = datasets.ImageFolder(val_dir, transform=data_transforms['val'])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ====== Load VGG16 ======
model = models.vgg16(pretrained=True)
model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
model = model.to(device)

# ====== Loss dan Optimizer ======
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# ====== Training ======
print("[INFO] Training started...")
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

# ====== Simpan Model ======
torch.save(model.state_dict(), model_save_path)
print(f"[INFO] Model disimpan di: {model_save_path}")

# ====== Evaluasi ======
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in val_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

# ====== Classification Report ======
print("\n[INFO] Classification Report:")
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
print(classification_report(y_true, y_pred, target_names=class_names))

# ====== Confusion Matrix ======
cm = confusion_matrix(y_true, y_pred)

# ====== Simpan ke Excel ======
with pd.ExcelWriter(excel_save_path) as writer:
    # Classification Report
    df_report = pd.DataFrame(report).transpose()
    df_report.to_excel(writer, sheet_name='Classification Report')

    # Confusion Matrix
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
    df_cm.to_excel(writer, sheet_name='Confusion Matrix')

print(f"[INFO] Hasil evaluasi disimpan di: {excel_save_path}")

# ====== Plot Confusion Matrix ======
plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - VGG16")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ====== Tambahan: Direktori Uji ======
test_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val1"

# ====== Dataset Uji ======
test_dataset = datasets.ImageFolder(test_dir, transform=data_transforms['val'])
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# ====== Evaluasi Data Uji ======
model.eval()
y_true_test = []
y_pred_test = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        y_true_test.extend(labels.numpy())
        y_pred_test.extend(preds.cpu().numpy())

# ====== Classification Report Test ======
print("\n[INFO] Classification Report (Test Data):")
report_test = classification_report(
    y_true_test, y_pred_test, target_names=class_names, output_dict=True)
print(classification_report(y_true_test, y_pred_test, target_names=class_names))

# ====== Confusion Matrix Test ======
cm_test = confusion_matrix(y_true_test, y_pred_test)

# ====== Simpan Tambahan ke Excel ======
with pd.ExcelWriter(excel_save_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    # Sheet: Classification Report Test
    df_report_test = pd.DataFrame(report_test).transpose()
    df_report_test.to_excel(writer, sheet_name='Classification Report - Test')

    # Sheet: Confusion Matrix Test
    df_cm_test = pd.DataFrame(cm_test, index=class_names, columns=class_names)
    df_cm_test.to_excel(writer, sheet_name='Confusion Matrix - Test')

print(f"[INFO] Hasil evaluasi TEST disimpan di: {excel_save_path}")

# ====== Plot Confusion Matrix Test ======
plt.figure(figsize=(14, 12))
sns.heatmap(cm_test, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Greens')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - VGG16 (Test Data)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


ALEXNET+PCA+SVM

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ========== Setup ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device yang digunakan:", device)

train_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/train"
val_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val"
batch_size = 32
class_names = sorted(os.listdir(train_dir))

# Transform & Dataloader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
train_loader = DataLoader(datasets.ImageFolder(train_dir, transform=transform), batch_size=batch_size, shuffle=False)
val_loader = DataLoader(datasets.ImageFolder(val_dir, transform=transform), batch_size=batch_size, shuffle=False)

# ========== Load AlexNet Feature Extractor ==========
model = models.alexnet(pretrained=True)
model.classifier = nn.Identity()  # Remove FC layer
model = model.to(device).eval()

# ========== Ekstraksi Fitur ==========
def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            outputs = model(imgs).cpu().numpy()
            feats.append(outputs)
            labels.extend(lbls.numpy())
    return np.concatenate(feats), np.array(labels)

X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)

# ========== PCA ==========
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

# ========== Train SVM ==========
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train_pca, y_train)

# ========== Simpan Model ==========
# Setelah fit PCA
joblib.dump(pca, "pca_alexnet.pkl")
joblib.dump(svm, "svm_pca_alexnet.pkl")
torch.save(model.state_dict(), "alexnet_feature_extractor.pth")

# ========== Evaluasi ==========
y_pred = svm.predict(X_val_pca)
acc = accuracy_score(y_val, y_pred)
print(f"\n🎯 Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=class_names))

# ========== Confusion Matrix ==========
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix - PCA + SVM (AlexNet)")
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

# ========== Uji Model dengan Data Test ==========
test_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val1"
test_loader = DataLoader(datasets.ImageFolder(test_dir, transform=transform), batch_size=batch_size, shuffle=False)

# Ekstraksi fitur data test
X_test, y_test = extract_features(test_loader)

# Transformasi PCA dan prediksi SVM
X_test_pca = pca.transform(X_test)
y_pred_test = svm.predict(X_test_pca)

# Evaluasi
acc_test = accuracy_score(y_test, y_pred_test)
print(f"\n✅ Test Accuracy: {acc_test:.4f}")
print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred_test, target_names=class_names))

# Confusion Matrix Test
cm_test = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix - PCA + SVM (AlexNet) - Test")
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()


RESNET18+PCA+SVM

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ========== Setup ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device yang digunakan:", device)

train_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/train"
val_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val"
batch_size = 32
class_names = sorted(os.listdir(train_dir))

# Transform & Dataloader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
train_loader = DataLoader(datasets.ImageFolder(train_dir, transform=transform), batch_size=batch_size, shuffle=False)
val_loader = DataLoader(datasets.ImageFolder(val_dir, transform=transform), batch_size=batch_size, shuffle=False)

# ========== Load ResNet18 sebagai Feature Extractor ==========
model = models.resnet18(pretrained=True)
model.fc = nn.Identity()  # Hapus classifier, jadikan feature extractor
model = model.to(device).eval()

# ========== Ekstraksi Fitur ==========
def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            outputs = model(imgs).cpu().numpy()
            feats.append(outputs)
            labels.extend(lbls.numpy())
    return np.concatenate(feats), np.array(labels)

X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)

# ========== PCA ==========
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

# ========== Train SVM ==========
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train_pca, y_train)

# ========== Simpan Model ==========
joblib.dump(pca, "pca_resnet18.pkl")
joblib.dump(svm, "svm_pca_resnet18.pkl")
torch.save(model.state_dict(), "resnet18_feature_extractor.pth")

# ========== Evaluasi ==========
y_pred = svm.predict(X_val_pca)
acc = accuracy_score(y_val, y_pred)
print(f"\n🎯 Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=class_names))

# ========== Confusion Matrix ==========
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix - PCA + SVM (ResNet18)")
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()

# ========== Evaluasi pada Data Test ==========
test_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val1"
test_loader = DataLoader(datasets.ImageFolder(test_dir, transform=transform), batch_size=batch_size, shuffle=False)

# Ekstraksi fitur test
X_test, y_test = extract_features(test_loader)

# Transformasi PCA dan prediksi
X_test_pca = pca.transform(X_test)
y_pred_test = svm.predict(X_test_pca)

# Akurasi dan Classification Report
acc_test = accuracy_score(y_test, y_pred_test)
print(f"\n🧪 Test Accuracy: {acc_test:.4f}")
print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred_test, target_names=class_names))

# Confusion Matrix
cm_test = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Confusion Matrix - PCA + SVM (ResNet18) - Test Data")
plt.xticks(rotation=90); plt.yticks(rotation=0)
plt.tight_layout(); plt.show()


VGGNET18+PCA+SVM

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ========== Setup ==========
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device yang digunakan:", device)

train_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/train"
val_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val"
batch_size = 32
class_names = sorted(os.listdir(train_dir))

# Transform & Dataloader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
train_loader = DataLoader(datasets.ImageFolder(train_dir, transform=transform), batch_size=batch_size, shuffle=False)
val_loader = DataLoader(datasets.ImageFolder(val_dir, transform=transform), batch_size=batch_size, shuffle=False)

# ========== Load VGG16 sebagai Feature Extractor ==========
model = models.vgg16(pretrained=True)
model.classifier[6] = nn.Identity()  # Hapus FC terakhir agar jadi feature extractor
model = model.to(device).eval()

# ========== Ekstraksi Fitur ==========
def extract_features(loader):
    feats, labels = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            outputs = model(imgs).cpu().numpy()
            feats.append(outputs)
            labels.extend(lbls.numpy())
    return np.concatenate(feats), np.array(labels)

X_train, y_train = extract_features(train_loader)
X_val, y_val = extract_features(val_loader)

# ========== PCA ==========
pca = PCA(n_components=0.95)
X_train_pca = pca.fit_transform(X_train)
X_val_pca = pca.transform(X_val)

# ========== Train SVM ==========
svm = SVC(kernel='rbf', C=10, gamma='scale')
svm.fit(X_train_pca, y_train)

# ========== Simpan Model ==========
joblib.dump(pca, "pca_vggnet16.pkl")
joblib.dump(svm, "svm_pca_vgg16.pkl")
torch.save(model.state_dict(), "vgg16_feature_extractor.pth")

# ========== Evaluasi ==========
y_pred = svm.predict(X_val_pca)
acc = accuracy_score(y_val, y_pred)
print(f"\n🎯 Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(classification_report(y_val, y_pred, target_names=class_names))

# ========== Confusion Matrix ==========
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - PCA + SVM (VGG16)")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ========== Evaluasi pada Data Test ==========
test_dir = "D:/Kuliah/Skripsi/Data_Preprocessed/val1"
test_loader = DataLoader(datasets.ImageFolder(test_dir, transform=transform), batch_size=batch_size, shuffle=False)

# Ekstraksi fitur test
X_test, y_test = extract_features(test_loader)

# Transformasi PCA dan prediksi
X_test_pca = pca.transform(X_test)
y_pred_test = svm.predict(X_test_pca)

# Akurasi dan Classification Report
acc_test = accuracy_score(y_test, y_pred_test)
print(f"\n🧪 Test Accuracy: {acc_test:.4f}")
print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred_test, target_names=class_names))

# Confusion Matrix
cm_test = confusion_matrix(y_test, y_pred_test)
plt.figure(figsize=(12, 10))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix - PCA + SVM (VGG16) - Test Data")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


INFERENCE TIME

In [ ]:
import torch
import time
from torchvision import transforms, models
from PIL import Image
import joblib
import numpy as np
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
import cv2
import pandas as pd
import os
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# ====== Fungsi bantu ======
def get_model_size(path):
    size_bytes = os.path.getsize(path)
    size_mb = size_bytes / (1024 * 1024)
    return round(size_mb, 2)

results = []

# ====== Path & Device ======
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
image_path = "D:/Kuliah/Skripsi/Script/Gita.jpg"

# Path model
finetuned_model_path = "efficientnetv2s_finetuned_jkt48.pth"
extractor_finetuned_path = "efficientnet_extractor_finetuned.pth"
pca_svm_finetuned_path = "pca_svm_pipeline_finetuned.pkl"
label_encoder_path = "label_encoder_finetuned.pkl"

scratch_model_path = "efficientnetv2s_jkt48_from_scratch1.pth"
extractor_scratch_path = "efficientnetv2s_feature_extractor.pth"
pca_svm_scratch_path = "best_model_pca_svm.pkl"

# ====== Preprocessing ======
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])
img = Image.open(image_path).convert("RGB")
input_tensor = transform(img).unsqueeze(0).to(device)

# ====== 1. EfficientNetV2-S Fine-tuned ======
model_finetuned = torch.load(finetuned_model_path, map_location=device)
model_finetuned.eval().to(device)

with torch.no_grad():
    for _ in range(3):
        _ = model_finetuned(input_tensor)
    start = time.time()
    for _ in range(10):
        output = model_finetuned(input_tensor)
    end = time.time()

t1 = (end - start) / 10
pred_class = output.argmax(dim=1).item()
label_encoder = joblib.load(label_encoder_path)
results.append({
    "Model": "EfficientNetV2-S Fine-tuned",
    "Inference Time (s)": round(t1, 6),
    "Accuracy": "-",
    "Predicted Class": label_encoder.inverse_transform([output.argmax(dim=1).item()])[0],
    "Model Size (MB)": get_model_size(finetuned_model_path),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [EfficientNetV2-S Fine-tuned]: {t1:.6f} detik")

# ====== 2. EfficientNetV2-S + PCA + SVM (pretrained) ======
extractor = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
extractor.classifier = torch.nn.Identity()
extractor.load_state_dict(torch.load(extractor_finetuned_path, map_location=device))
extractor.eval().to(device)

pca_svm_pipeline = joblib.load(pca_svm_finetuned_path)

with torch.no_grad():
    for _ in range(3):
        _ = extractor(input_tensor)
    start = time.time()
    for _ in range(10):
        feat = extractor(input_tensor).cpu().numpy()
        feat_pca = pca_svm_pipeline["pca"].transform(feat)
        pred = pca_svm_pipeline["svm"].predict(feat_pca)
        label = label_encoder.inverse_transform(pred)
    end = time.time()

t2 = (end - start) / 10
results.append({
    "Model": "EfficientNetV2-S + PCA + SVM (pretrained)",
    "Inference Time (s)": round(t2, 6),
    "Accuracy": "-",
    "Predicted Class": label[0],  # Sudah string
    "Model Size (MB)": round(
        get_model_size(extractor_finetuned_path) +
        get_model_size(pca_svm_finetuned_path) +
        get_model_size(label_encoder_path), 2),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [EfficientNetV2-S + PCA + SVM]: {t2:.6f} detik")

# ====== 3. EfficientNetV2-S From Scratch ======
model_scratch = efficientnet_v2_s(weights=None)
model_scratch.classifier = torch.nn.Linear(1280, len(label_encoder.classes_))
model_scratch.load_state_dict(torch.load(scratch_model_path, map_location=device))
model_scratch.eval().to(device)

with torch.no_grad():
    for _ in range(3):
        _ = model_scratch(input_tensor)
    start = time.time()
    for _ in range(10):
        output = model_scratch(input_tensor)
    end = time.time()

t3 = (end - start) / 10
pred_class = output.argmax(dim=1).item()
results.append({
    "Model": "EfficientNetV2-S From Scratch",
    "Inference Time (s)": round(t3, 6),
    "Accuracy": "-",
    "Predicted Class": label_encoder.inverse_transform([output.argmax(dim=1).item()])[0],
    "Model Size (MB)": get_model_size(scratch_model_path),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [EfficientNetV2-S From Scratch]: {t3:.6f} detik")

# ====== 4. From Scratch + PCA + SVM ======
extractor_scratch = efficientnet_v2_s(weights=None)
extractor_scratch.classifier = torch.nn.Identity()
extractor_scratch.load_state_dict(torch.load(extractor_scratch_path, map_location=device))
extractor_scratch.eval().to(device)

scratch_pipeline = joblib.load(pca_svm_scratch_path)

with torch.no_grad():
    for _ in range(3):
        _ = extractor_scratch(input_tensor)
    start = time.time()
    for _ in range(10):
        feat = extractor_scratch(input_tensor).cpu().numpy()
        feat_pca = scratch_pipeline["pca"].transform(feat)
        pred = scratch_pipeline["svm"].predict(feat_pca)
    end = time.time()

t4 = (end - start) / 10
results.append({
    "Model": "EfficientNetV2-S + PCA + SVM (from scratch)",
    "Inference Time (s)": round(t4, 6),
    "Accuracy": "-",
    "Predicted Class": label_encoder.inverse_transform(pred)[0],
    "Model Size (MB)": round(
        get_model_size(extractor_scratch_path) +
        get_model_size(pca_svm_scratch_path), 2),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [EfficientNetV2-S + PCA + SVM (from scratch)]: {t4:.6f} detik")
predicted_class = label_encoder.inverse_transform(pred)[0]
print(f"🎯 Predicted Class: {predicted_class}")

# ====== 5. Eigenfaces ======
pca_eigen = joblib.load("model_pca_eigenfaces.pkl")
svm_eigen = joblib.load("model_svm_eigenfaces.pkl")
le_eigen = joblib.load("label_encoder.pkl")

img = load_img(image_path, target_size=(128, 128))
img_array = img_to_array(img).astype("uint8")
img_gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
img_flat = img_gray.flatten().reshape(1, -1)

for _ in range(3):
    _ = pca_eigen.transform(img_flat)
start = time.time()
for _ in range(10):
    feat = pca_eigen.transform(img_flat)
    pred = svm_eigen.predict(feat)
    label = le_eigen.inverse_transform(pred)
end = time.time()

t5 = (end - start) / 10
results.append({
    "Model": "Eigenfaces (PCA + SVM)",
    "Inference Time (s)": round(t5, 6),
    "Accuracy": "-",
    "Predicted Class": label[0],  # Sudah nama langsung
    "Model Size (MB)": round(
        get_model_size("model_pca_eigenfaces.pkl") +
        get_model_size("model_svm_eigenfaces.pkl") +
        get_model_size("label_encoder.pkl"), 2),
    "Input Size": "128x128 (grayscale)"
})
print(f"✅ Inference Time [Eigenfaces]: {t5:.6f} detik")

# ====== 6. Fisherfaces ======
fisher_model = joblib.load("fisherfaces_model.pkl")
pca_fisher = fisher_model['pca']
lda_fisher = fisher_model['lda']
svm_fisher = fisher_model['svm']

feat_pca = pca_fisher.transform(img_flat)
feat_lda = lda_fisher.transform(feat_pca)

for _ in range(3):
    _ = svm_fisher.predict(feat_lda)
start = time.time()
for _ in range(10):
    pred = svm_fisher.predict(feat_lda)
end = time.time()

t6 = (end - start) / 10
results.append({
    "Model": "Fisherfaces (PCA + LDA + SVM)",
    "Inference Time (s)": round(t6, 6),
    "Accuracy": "-",
    "Predicted Class": pred[0],  # Sudah nama langsung
    "Model Size (MB)": get_model_size("fisherfaces_model.pkl"),
    "Input Size": "128x128 (grayscale)"
})

print(f"✅ Inference Time [Fisherfaces]: {t6:.6f} detik")

# ====== 7. AlexNet ======
class_names = sorted(os.listdir("D:/Kuliah/Skripsi/Data_Preprocessed/train"))
alexnet = models.alexnet(pretrained=True)
alexnet.classifier[6] = torch.nn.Linear(alexnet.classifier[6].in_features, len(class_names))
alexnet.load_state_dict(torch.load("alexnet_model.pth"))
alexnet.to(device).eval()

img_tensor = transform(img).unsqueeze(0).to(device)
for _ in range(3):
    _ = alexnet(img_tensor)
start = time.time()
for _ in range(10):
    with torch.no_grad():
        out = alexnet(img_tensor)
        _, pred = torch.max(out, 1)
end = time.time()

t7 = (end - start) / 10
results.append({
    "Model": "AlexNet (Pretrained + Fine-tuned)",
    "Inference Time (s)": round(t7, 6),
    "Accuracy": "-",
    "Predicted Class": class_names[pred.item()],
    "Model Size (MB)": get_model_size("alexnet_model.pth"),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [AlexNet]: {t7:.6f} detik")

# ====== 8. ResNet18 ======
resnet18 = models.resnet18(pretrained=True)
resnet18.fc = torch.nn.Linear(resnet18.fc.in_features, len(class_names))
resnet18.load_state_dict(torch.load("resnet18_model.pth"))
resnet18.to(device).eval()

for _ in range(3):
    _ = resnet18(img_tensor)
start = time.time()
for _ in range(10):
    with torch.no_grad():
        out = resnet18(img_tensor)
        _, pred = torch.max(out, 1)
end = time.time()

t8 = (end - start) / 10
results.append({
    "Model": "ResNet18 (Pretrained + Fine-tuned)",
    "Inference Time (s)": round(t8, 6),
    "Accuracy": "-",
    "Predicted Class": class_names[pred.item()],
    "Model Size (MB)": get_model_size("resnet18_model.pth"),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [ResNet18]: {t8:.6f} detik")

# ====== 9. VGG16 ======
from torchvision.models import vgg16
vgg = vgg16(pretrained=True)
vgg.classifier[6] = torch.nn.Linear(vgg.classifier[6].in_features, len(class_names))
vgg.load_state_dict(torch.load("vgg16_model.pth"))
vgg.to(device).eval()

for _ in range(3):
    _ = vgg(img_tensor)
start = time.time()
for _ in range(10):
    with torch.no_grad():
        out = vgg(img_tensor)
        _, pred = torch.max(out, 1)
end = time.time()

t9 = (end - start) / 10
results.append({
    "Model": "VGG16 (Pretrained + Fine-tuned)",
    "Inference Time (s)": round(t9, 6),
    "Accuracy": "-",
    "Predicted Class": class_names[pred.item()],
    "Model Size (MB)": get_model_size("vgg16_model.pth"),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [VGG16]: {t9:.6f} detik")

# ====== 10. PCA + SVM (AlexNet Feature Extractor) ======
model_alexnet = models.alexnet(pretrained=True)  # pretrained=True untuk konsistensi layer awal
model_alexnet.classifier = nn.Identity()  # Hilangkan FC layer
model_alexnet.load_state_dict(torch.load("alexnet_feature_extractor.pth", map_location=device))
model_alexnet = model_alexnet.to(device).eval()

# Load PCA dan SVM secara terpisah
pca_alexnet = joblib.load("pca_alexnet.pkl")
svm_alexnet = joblib.load("svm_pca_alexnet.pkl")

# Ambil ulang citra RGB
img_rgb = Image.open(image_path).convert("RGB")
input_tensor_alex = transform(img_rgb).unsqueeze(0).to(device)

# Inference Time: Ekstraksi Fitur
with torch.no_grad():
    for _ in range(3):
        _ = model_alexnet(input_tensor_alex)

    start_feat = time.time()
    feat = model_alexnet(input_tensor_alex).cpu().numpy()
    end_feat = time.time()

    # Transformasi PCA
    start_pca = time.time()
    feat_pca = pca_alexnet.transform(feat)
    end_pca = time.time()

    # Prediksi SVM
    start_svm = time.time()
    pred = svm_alexnet.predict(feat_pca)
    end_svm = time.time()

total_time = end_svm - start_feat

results.append({
    "Model": "AlexNet + PCA + SVM",
    "Inference Time (s)": round(total_time, 6),
    "Accuracy": "-",
    "Predicted Class": class_names[pred[0]],
    "Model Size (MB)": round(
        get_model_size("alexnet_feature_extractor.pth") +
        get_model_size("pca_alexnet.pkl") +
        get_model_size("svm_pca_alexnet.pkl"), 2),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [AlexNet + PCA + SVM]: {total_time:.6f} detik")

# ====== 11. ResNet18 + PCA + SVM ======
resnet_extractor = models.resnet18(pretrained=True)
resnet_extractor.fc = nn.Identity()
resnet_extractor.load_state_dict(torch.load("resnet18_feature_extractor.pth", map_location=device))
resnet_extractor = resnet_extractor.to(device).eval()

# Load PCA dan SVM
pca_resnet = joblib.load("pca_resnet18.pkl")
svm_resnet = joblib.load("svm_pca_resnet18.pkl")

# Preprocess gambar
img_resnet = transform(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)

with torch.no_grad():
    for _ in range(3):
        _ = resnet_extractor(img_resnet)

    start_feat = time.time()
    feat = resnet_extractor(img_resnet).cpu().numpy()
    end_feat = time.time()

    start_pca = time.time()
    feat_pca = pca_resnet.transform(feat)
    end_pca = time.time()

    start_svm = time.time()
    pred = svm_resnet.predict(feat_pca)
    end_svm = time.time()

total_time_resnet = end_svm - start_feat

results.append({
    "Model": "ResNet18 + PCA + SVM",
    "Inference Time (s)": round(total_time_resnet, 6),
    "Accuracy": "-",
    "Predicted Class": class_names[pred[0]],
    "Model Size (MB)": round(
        get_model_size("resnet18_feature_extractor.pth") +
        get_model_size("pca_resnet18.pkl") +
        get_model_size("svm_pca_resnet18.pkl"), 2),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [ResNet18 + PCA + SVM]: {total_time_resnet:.6f} detik")


# ====== 12. VGG16 + PCA + SVM ======
vgg16_extractor = models.vgg16(pretrained=True)
vgg16_extractor.classifier[6] = nn.Identity()
vgg16_extractor.load_state_dict(torch.load("vgg16_feature_extractor.pth", map_location=device))
vgg16_extractor.to(device).eval()

# Load PCA dan SVM
pca_vgg16 = joblib.load("pca_vggnet16.pkl")
svm_vgg16 = joblib.load("svm_pca_vgg16.pkl")

# Preprocessing gambar
img_vgg16 = transform(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)

with torch.no_grad():
    for _ in range(3):
        _ = vgg16_extractor(img_vgg16)

    start_feat = time.time()
    feat = vgg16_extractor(img_vgg16).cpu().numpy()
    end_feat = time.time()

    start_pca = time.time()
    feat_pca = pca_vgg16.transform(feat)
    end_pca = time.time()

    start_svm = time.time()
    pred = svm_vgg16.predict(feat_pca)
    end_svm = time.time()

total_time_vgg16 = end_svm - start_feat

results.append({
    "Model": "VGG16 + PCA + SVM",
    "Inference Time (s)": round(total_time_vgg16, 6),
    "Accuracy": "-",
    "Predicted Class": class_names[pred[0]],
    "Model Size (MB)": round(
        get_model_size("vgg16_feature_extractor.pth") +
        get_model_size("pca_vggnet16.pkl") +
        get_model_size("svm_pca_vgg16.pkl"), 2),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [VGG16 + PCA + SVM]: {total_time_vgg16:.6f} detik")

# ====== 13. From Scratch + SVM (tanpa PCA) ======
import torch
import joblib
import time
import numpy as np
from torchvision.models import efficientnet_v2_s

# Load model feature extractor (tanpa classifier)
extractor_scratch = efficientnet_v2_s(weights=None)
extractor_scratch.classifier = torch.nn.Identity()
extractor_scratch.load_state_dict(torch.load("efficientnetv2s_feature_extractor.pth", map_location=device))
extractor_scratch.eval().to(device)

# Load SVM model tanpa PCA
svm_model = joblib.load("svm_efficientnetv2s_tanpa_pca.pkl")

# Inference 10x untuk estimasi waktu rata-rata
with torch.no_grad():
    for _ in range(3):
        _ = extractor_scratch(input_tensor)  # Warm-up
    start = time.time()
    for _ in range(10):
        feat = extractor_scratch(input_tensor).cpu().numpy()  # shape: (1, 1280)
        pred = svm_model.predict(feat)  # langsung prediksi tanpa PCA
    end = time.time()

t4 = (end - start) / 10
results.append({
    "Model": "EfficientNetV2-S + SVM (tanpa PCA)",
    "Inference Time (s)": round(t4, 6),
    "Accuracy": "-",  # isi jika punya nilai akurasi
    "Predicted Class": label_encoder.inverse_transform(pred)[0],  # pastikan label_encoder sudah ada
    "Model Size (MB)": round(
        get_model_size("efficientnetv2s_feature_extractor.pth") +
        get_model_size("svm_efficientnetv2s_tanpa_pca.pkl"), 2),
    "Input Size": "224x224"
})
print(f"✅ Inference Time [EfficientNetV2-S + SVM (tanpa PCA)]: {t4:.6f} detik")

# ====== Simpan Excel ======
df = pd.DataFrame(results)
df.to_excel("inference_results.xlsx", index=False)
print("\n✅ Hasil disimpan ke 'inference_results.xlsx'")